In [1]:
import numpy as np
import pandas as pd 
from sklearn.preprocessing import RobustScaler

In [2]:
# 1) Load  & inspect
csv_path = "dataset/raw_loan_dataset.csv"
df = pd.read_csv(csv_path)
print("\n=== initial HEAD ===")
print(df.head())
print("\n=== INITIAL INFO ===")
print(df.info())

print("\n=== INITIAL MISSING VALUES ===")
print(df.isnull().sum())


=== initial HEAD ===
    Income  CreditScore  EmploymentYears LoanAmount HasCollateral  \
0   108810        537.0              1.1      25800           Yes   
1    96482        524.0             15.0      11200             Y   
2    28478          NaN              5.4      12100            No   
3  $25,851        561.0             17.6       7000           Yes   
4    38396        527.0              9.8      10700            No   

  PreviousDefaults  Approved  
0               No        No  
1               No       Yes  
2               No       Yes  
3               No       Yes  
4               No  Approved  

=== INITIAL INFO ===
<class 'pandas.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            98 non-null     str    
 1   CreditScore       97 non-null     float64
 2   EmploymentYears   98 non-null     float64
 3   LoanAmount        98 n

In [3]:
# 2) Clean currency formatting

df["Income"] = df["Income"].replace(r"[\$,]","",regex=True).astype(float)
df["LoanAmount"] = df["LoanAmount"].replace(r"[\$,]", "", regex=True).astype(float)

In [4]:
# 3) Normalize categorical typos BEFORE imputation
# --------------------------------
yes_no_map = {
    "yes": "Yes", "y": "Yes", "yse": "Yes", "1": "Yes", "approved": "Yes",
    "no": "No", "n": "No", "0": "No", "rejected": "No",
}

df["HasCollateral"] = df["HasCollateral"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})
df["PreviousDefaults"] = df["PreviousDefaults"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})
df["Approved"] = df["Approved"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})

print("\n=== after  normalize head ===")
print(df.head())


=== after  normalize head ===
     Income  CreditScore  EmploymentYears  LoanAmount HasCollateral  \
0  108810.0        537.0              1.1     25800.0           Yes   
1   96482.0        524.0             15.0     11200.0           Yes   
2   28478.0          NaN              5.4     12100.0            No   
3   25851.0        561.0             17.6      7000.0           Yes   
4   38396.0        527.0              9.8     10700.0            No   

  PreviousDefaults Approved  
0               No       No  
1               No      Yes  
2               No      Yes  
3               No      Yes  
4               No      Yes  


In [5]:
# --------------------------------
# 4) Impute missing values

df["Income"] = df["Income"].fillna(df["Income"]).median()
df["Income"] = df["Income"].fillna(df["Income"].median())
df["CreditScore"] = df["CreditScore"].fillna(df["CreditScore"].median())
df["EmploymentYears"] = df["EmploymentYears"].fillna(df["EmploymentYears"].median())
df["LoanAmount"] = df["LoanAmount"].fillna(df["LoanAmount"].median())
df["HasCollateral"] = df["HasCollateral"].fillna(df["HasCollateral"].mode()[0])
df["PreviousDefaults"] = df["PreviousDefaults"].fillna(df["PreviousDefaults"].mode()[0])
print("\n=== after MISSING VALUES ===")
print(df.isnull().sum())
print("\n=== after HEAD ===")
print(df.head())



=== after MISSING VALUES ===
Income              0
CreditScore         0
EmploymentYears     0
LoanAmount          0
HasCollateral       0
PreviousDefaults    0
Approved            0
dtype: int64

=== after HEAD ===
    Income  CreditScore  EmploymentYears  LoanAmount HasCollateral  \
0  69460.5        537.0              1.1     25800.0           Yes   
1  69460.5        524.0             15.0     11200.0           Yes   
2  69460.5        638.0              5.4     12100.0            No   
3  69460.5        561.0             17.6      7000.0           Yes   
4  69460.5        527.0              9.8     10700.0            No   

  PreviousDefaults Approved  
0               No       No  
1               No      Yes  
2               No      Yes  
3               No      Yes  
4               No      Yes  


In [6]:
# 5) Remove duplicates
# --------------------------------
before = df.shape[0]
df = df.drop_duplicates()
print(f"\nDropped duplicates: {before} -> {df.shape[0]} rows")


Dropped duplicates: 103 -> 100 rows


In [7]:
 # 6) IQR capping on numeric columns
# L3: clip extreme values to the IQR fence — same idea as house data-processing.py
# --------------------------------
def iqr_fun(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper

low_income,  high_income  = iqr_fun(df["Income"])
low_credit,  high_credit  = iqr_fun(df["CreditScore"])
low_loan,    high_loan    = iqr_fun(df["LoanAmount"])
low_emp,     high_emp     = iqr_fun(df["EmploymentYears"])

df["Income"]          = df["Income"].clip(lower=low_income,  upper=high_income)
df["CreditScore"]     = df["CreditScore"].clip(lower=low_credit, upper=high_credit)
df["LoanAmount"]      = df["LoanAmount"].clip(lower=low_loan,    upper=high_loan)
df["EmploymentYears"] = df["EmploymentYears"].clip(lower=low_emp,     upper=high_emp)


In [8]:
# 7) Label encoding (Yes/No → 0/1)
# L3: same mapping for target (Approved) and two-category features
# Approved=1, Rejected=0
# --------------------------------
df["Approved"] = df["Approved"].map({"Yes": 1, "No": 0}).astype(int)

print("\n=== CLASS DISTRIBUTION (after label encoding) ===")
print(df["Approved"].value_counts())
print(df["Approved"].value_counts(normalize=True).round(3))


=== CLASS DISTRIBUTION (after label encoding) ===
Approved
1    66
0    34
Name: count, dtype: int64
Approved
1    0.66
0    0.34
Name: proportion, dtype: float64


In [9]:
# 8) Class balance check
# L3/L5: imbalanced classes can make Accuracy misleading
# --------------------------------
class_ratio = df["Approved"].value_counts(normalize=True).min()
if class_ratio < 0.30:
    print("\nWarning: severely imbalanced classes — consider balancing techniques (L3).")
else:
    print("\nClass balance OK for baseline Accuracy (both classes well represented).")

df["HasCollateral"] = df["HasCollateral"].map({"Yes": 1, "No": 0}).astype(int)
df["PreviousDefaults"] = df["PreviousDefaults"].map({"Yes": 1, "No": 0}).astype(int)


Class balance OK for baseline Accuracy (both classes well represented).


In [10]:

# 9) Feature Engineering
# --------------------------------

# Debt-to-Income Ratio
df["DebtToIncome"] = df["LoanAmount"] / df["Income"].replace(0, np.nan)

# Income per Year Employed
df["IncomePerYearEmployed"] = df["Income"] / (df["EmploymentYears"] + 1)

print("\n=== FEATURE ENGINEERING ===")
print(df[["DebtToIncome", "IncomePerYearEmployed"]].head())




=== FEATURE ENGINEERING ===
   DebtToIncome  IncomePerYearEmployed
0      0.371434           33076.428571
1      0.161243            4341.281250
2      0.174200           10853.203125
3      0.100777            3734.435484
4      0.154044            6431.527778


In [11]:
# --------------------------------
# 10) Feature Scaling using RobustScaler
# --------------------------------

binary_cols = ["Approved", "HasCollateral", "PreviousDefaults"]

numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns

scale_cols = [col for col in numeric_cols if col not in binary_cols]

scaler = RobustScaler()

df[scale_cols] = scaler.fit_transform(df[scale_cols])

print("\n=== ROBUST SCALER APPLIED ===")
print(df.head())


=== ROBUST SCALER APPLIED ===
   Income  CreditScore  EmploymentYears  LoanAmount  HasCollateral  \
0     0.0    -0.653722        -0.978632    0.246080              1   
1     0.0    -0.737864         0.209402   -0.458384              1   
2     0.0     0.000000        -0.611111   -0.414958              0   
3     0.0    -0.498382         0.431624   -0.661037              1   
4     0.0    -0.718447        -0.235043   -0.482509              0   

   PreviousDefaults  Approved  DebtToIncome  IncomePerYearEmployed  
0                 0         0      0.246080               4.748084  
1                 0         1     -0.458384              -0.133345  
2                 0         1     -0.414958               0.972878  
3                 0         1     -0.661037              -0.236434  
4                 0         1     -0.482509               0.221739  


In [12]:
# --------------------------------
# 11) Final snapshot+ save
# Features (X) = all columns except Approved; label (y) = Approved
# --------------------------------
print("\n=== FINAL HEAD ===")
print(df.head())

print("\n=== FINAL INFO ===")
print(df.info())

print("\n=== FINAL MISSING VALUES ===")
print(df.isnull().sum())

OUT_PATH = "dataset/clean_loan_preprocessing"

df.to_csv(OUT_PATH, index=False)
print(f"\nSaved cleaned dataset to {OUT_PATH}")


=== FINAL HEAD ===
   Income  CreditScore  EmploymentYears  LoanAmount  HasCollateral  \
0     0.0    -0.653722        -0.978632    0.246080              1   
1     0.0    -0.737864         0.209402   -0.458384              1   
2     0.0     0.000000        -0.611111   -0.414958              0   
3     0.0    -0.498382         0.431624   -0.661037              1   
4     0.0    -0.718447        -0.235043   -0.482509              0   

   PreviousDefaults  Approved  DebtToIncome  IncomePerYearEmployed  
0                 0         0      0.246080               4.748084  
1                 0         1     -0.458384              -0.133345  
2                 0         1     -0.414958               0.972878  
3                 0         1     -0.661037              -0.236434  
4                 0         1     -0.482509               0.221739  

=== FINAL INFO ===
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column                 Non-N